In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

from data_loader import load_dataset
from preprocessing import split_features, build_preprocessor
from config import RANDOM_STATE, TEST_SIZE, MODEL_DIR

## LASSO Regression (L1 Regularisation)

LASSO (Least Absolute Shrinkage and Selection Operator) adds an L1 penalty to the
ordinary least-squares loss:

$$\text{Loss} = \text{MSE} + \alpha \sum_{j} |w_j|$$

The key properties of L1 regularisation:
- **Sparsity**: L1 pushes coefficients to *exactly* zero, effectively performing feature selection.
- **`alpha`**: controls regularisation strength. Higher → more coefficients zeroed out. `alpha=0` recovers plain linear regression.
- Use **cross-validation** (`LassoCV`) in production to tune `alpha` automatically.

In [1]:
X, y, df = load_dataset()

print("X shape:", X.shape)
print("y shape:", y.shape)

NameError: name 'load_dataset' is not defined

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

In [ ]:
numeric_features, categorical_features = split_features(X_train)

In [ ]:
preprocessor = build_preprocessor(
    numeric_features,
    categorical_features
)

In [ ]:
from sklearn.pipeline import Pipeline

ALPHA = 1.0  # L1 regularisation strength — tune this

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Lasso(alpha=ALPHA, max_iter=10_000, random_state=RANDOM_STATE))
])

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_valid)

mae = mean_absolute_error(y_valid, y_pred)

print(f"LASSO Regression MAE (alpha={ALPHA}):", mae)

In [ ]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_valid)

print("Dummy MAE:", mean_absolute_error(y_valid, dummy_pred))

## Inspect feature sparsity

One of the main benefits of LASSO is that some coefficients are driven to exactly zero.
The cell below extracts feature names after preprocessing and shows which features were kept vs. zeroed out.

In [ ]:
# Retrieve feature names produced by the preprocessor
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients  = model.named_steps["regressor"].coef_

coef_df = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
)

n_total  = len(coef_df)
n_zeroed = (coef_df["coefficient"] == 0).sum()

print(f"Total features : {n_total}")
print(f"Zeroed out     : {n_zeroed}  ({100 * n_zeroed / n_total:.1f}%)")
print(f"Active features: {n_total - n_zeroed}")
print()
print("Top 10 features by |coefficient|:")
print(coef_df.head(10).to_string(index=False))

## Tune alpha with cross-validation (optional)

`LassoCV` tries a range of alpha values with k-fold CV and picks the best one automatically.
Swap it in instead of the manual `ALPHA` above when you want a more principled choice.

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline

cv_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LassoCV(cv=5, max_iter=10_000, random_state=RANDOM_STATE, n_jobs=-1))
])

cv_model.fit(X_train, y_train)

best_alpha = cv_model.named_steps["regressor"].alpha_
print("Best alpha found by CV:", best_alpha)

y_pred_cv = cv_model.predict(X_valid)
print("LASSO CV MAE:", mean_absolute_error(y_valid, y_pred_cv))

In [ ]:
joblib.dump(model, MODEL_DIR / "lasso_model.pkl")
joblib.dump(cv_model, MODEL_DIR / "lasso_cv_model.pkl")